# 07 Topic Modeling
Discover latent topics in customer reviews using LDA.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load cleaned reviews
df = pd.read_csv('../data/cleaned_reviews.csv')
texts = df['reviewText_clean']

print(f'Loaded {len(texts)} reviews for topic modeling')

In [ ]:
# Create document-term matrix
print('Creating document-term matrix...')
vectorizer = CountVectorizer(
    max_features=2000,
    min_df=5,
    max_df=0.8,
    stop_words='english'
)

doc_term_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

print(f'Document-term matrix shape: {doc_term_matrix.shape}')
print(f'Vocabulary size: {len(feature_names)}')

# Save vectorizer
joblib.dump(vectorizer, '../models/lda_vectorizer.joblib')
print('LDA vectorizer saved')

In [ ]:
# Train LDA model
print('Training LDA model with 5 topics...')
n_topics = 5

lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=20,
    learning_method='online',
    n_jobs=-1
)

lda.fit(doc_term_matrix)
print(f'LDA model trained')

# Save model
joblib.dump(lda, '../models/lda_model.joblib')
print('LDA model saved')

In [ ]:
# Display top words per topic
def display_topics(model, feature_names, n_top_words=10):
    topics_dict = {}
    
    for topic_idx, topic in enumerate(model.components_):
        top_word_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_word_indices]
        top_weights = topic[top_word_indices]
        
        topics_dict[f'Topic {topic_idx}'] = {
            'words': top_words,
            'weights': top_weights.tolist()
        }
        
        print(f'\nTopic {topic_idx}:')
        print('  ' + ', '.join(top_words))
    
    return topics_dict

topics = display_topics(lda, feature_names)

In [ ]:
# Topic distribution per document
doc_topic_dist = lda.transform(doc_term_matrix)

# Add dominant topic to dataframe
df['dominant_topic'] = doc_topic_dist.argmax(axis=1)
df['topic_probability'] = doc_topic_dist.max(axis=1)

print('\nTopic distribution:')
print(df['dominant_topic'].value_counts().sort_index())

print('\nAverage topic probability:')
print(f'{df["topic_probability"].mean():.3f}')

In [ ]:
# Visualize topic distribution
fig, ax = plt.subplots(figsize=(10, 5))

topic_counts = df['dominant_topic'].value_counts().sort_index()
ax.bar([f'Topic {i}' for i in topic_counts.index], topic_counts.values, color='skyblue', edgecolor='black')
ax.set_ylabel('Number of Documents')
ax.set_title('Document Distribution Across Topics')
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(topic_counts.values):
    ax.text(i, v + 10, str(v), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Topic-sentiment analysis
fig, ax = plt.subplots(figsize=(10, 5))

sentiment_by_topic = df.groupby('dominant_topic')['sentiment'].agg(['sum', 'count'])
sentiment_by_topic['percent_positive'] = (sentiment_by_topic['sum'] / sentiment_by_topic['count'] * 100)

ax.bar([f'Topic {i}' for i in sentiment_by_topic.index], 
        sentiment_by_topic['percent_positive'], 
        color=['red' if x < 50 else 'green' for x in sentiment_by_topic['percent_positive']])
ax.set_ylabel('% Positive Reviews')
ax.set_title('Sentiment by Topic')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save topic results
import json
from pathlib import Path

results = {
    'n_topics': n_topics,
    'topics': topics,
    'doc_topic_distribution': doc_topic_dist.tolist(),
    'dominant_topic_per_doc': df['dominant_topic'].tolist()
}

report_path = '../assets/reports/topic_modeling_report.json'
Path(report_path).parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)

print(f'Topic modeling results saved to {report_path}')